# 6 · Resume Screener
## Scoring Rubric + Comparable Structured Output Across Many Inputs
⏱ ~20 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/6-resume-screener/resume_screener_workbook.ipynb)

A recruiter posting a job opening gets 50 resumes. Reading each one carefully takes 5+ minutes. Most companies either screen too fast (missing good candidates) or too slow (creating a backlog).

This example builds an agent that applies a **consistent scoring rubric** to every resume and returns a `ResumeScore` — a typed object you can sort, filter, and rank directly in Python. No parsing, no inconsistency.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The screening problem — why consistency matters |
| 2 | Design the `ResumeScore` schema |
| 3 | Embed the job spec and rubric in the system prompt |
| 4 | Screen a single resume |
| 5 | Screen multiple resumes and rank them |
| ★ | Exercises + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab (run the install cell)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
if True:  # always runs
    try:
        from google.colab import userdata  # noqa: F401
        import subprocess
        subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
        print("Packages installed.")
    except ImportError:
        print("Local environment — install with: pip install langchain-openai pydantic python-dotenv")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
    print("API key loaded from .env.")

---

## Part 1 — The Screening Problem

---

Free-text screening breaks at scale:

```
Resume 1 → LLM → "This candidate has strong Python skills and would be a good fit."
Resume 2 → LLM → "Excellent background in backend development. 7/10 recommendation."
Resume 3 → LLM → "Meets most requirements. Some concerns about cloud experience."
```

Three different formats. Impossible to sort. Impossible to compare. One recruiter says 7/10, the next says "good fit" — no common scale.

**The solution:** one Pydantic schema applied to every resume.

```
Resume 1 → ResumeScore(score=8, tier="strong_yes", skills_matched=[...], ...)
Resume 2 → ResumeScore(score=5, tier="maybe",      skills_matched=[...], ...)
Resume 3 → ResumeScore(score=3, tier="no",         skills_matched=[...], ...)

# Now you can sort:
ranked = sorted(results, key=lambda r: r.overall_score, reverse=True)
```

Same schema, same scale, every time.

---

## Part 2 — The `ResumeScore` Schema

---

The schema captures what a recruiter actually needs:
- **Score 1-10** and a tier (strong_yes / yes / maybe / no)  
- **What matched** and **what's missing** against the job spec  
- **One standout signal** and **one main concern**  
- **Recommended action** — schedule / hold / pass

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field


class ResumeScore(BaseModel):
    candidate_name: str
    overall_score: int = Field(ge=1, le=10, description="Overall fit score 1-10")
    tier: Literal["strong_yes", "yes", "maybe", "no"]
    years_experience: int = Field(ge=0, description="Estimated years of relevant experience")
    skills_matched: List[str] = Field(description="Job spec skills found in the resume")
    skills_missing: List[str] = Field(description="Job spec skills not found in the resume")
    standout: str = Field(description="One sentence on the strongest signal in this resume")
    concern: str = Field(description="One sentence on the biggest gap or risk")
    recommended_action: Literal["schedule_interview", "hold_for_review", "pass"]

# Note: every field is either a Literal (constrained), a List[str] (parseable),
# or a plain str/int. The model can't return "Yes" instead of "strong_yes".
# The scoring rubric (what 8/10 means) lives in the system prompt, not here.
print("Schema defined.")

---

## Part 3 — Embedding the Rubric in the System Prompt

---

A schema tells the model *what shape to return*. The rubric tells it *how to score*.

Both are needed. Without the rubric, the model has no consistent frame for what a 7 vs 8 means. Without the schema, the rubric produces free text you can't sort.

The job spec goes in the system prompt — it's context the model needs for every resume it scores.

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

JOB_SPEC = """
Role: Senior Python Backend Engineer
Company: FinTech startup (Series B, 80 employees)

Required skills:
- Python 5+ years
- REST API design (FastAPI or Django REST)
- PostgreSQL or similar relational database
- Cloud deployment (AWS or GCP)
- Git, CI/CD pipelines

Nice to have:
- Financial domain experience
- Pydantic / data validation
- Docker / Kubernetes
- Team leadership or mentoring

Red flags:
- Less than 3 years total experience
- No production deployment experience
- Only frontend or data science work (we need backend)
"""

SYSTEM_PROMPT = SystemMessage(
    f"""You are a senior technical recruiter screening candidates for a backend engineering role.

Job Specification:
{JOB_SPEC}

Score the candidate 1-10 against the job spec:
- 9-10: exceptional fit, clear must-interview
- 7-8: strong match, schedule interview
- 5-6: partial match, review more carefully
- 3-4: significant gaps, likely pass
- 1-2: does not meet minimum requirements

Tier mapping:
  strong_yes  → score 8-10, all required skills present
  yes         → score 6-7, most required skills present
  maybe       → score 4-5, some required skills, notable gaps
  no          → score 1-3, does not meet requirements

Be honest. Do not inflate scores."""
)

print("System prompt defined.")

---

## Part 4 — Screen a Single Resume

---

`SYSTEM_PROMPT | llm.with_structured_output(ResumeScore)` is the complete screener.
The system prompt provides the rubric; the schema enforces the output shape.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
screener = SYSTEM_PROMPT | llm.with_structured_output(ResumeScore)

# Screen one resume
alex_resume = """Alex Kim — Senior Software Engineer
7 years Python. Built REST APIs with FastAPI at two fintech startups.
Led a team of 3 engineers. PostgreSQL, Redis, AWS (ECS, RDS).
Deployed to production weekly using GitHub Actions + Docker.
Currently at PayScale Inc, handling $2M/day transaction volume.
Skills: Python, FastAPI, PostgreSQL, AWS, Docker, Pydantic, Git."""

result = screener.invoke(alex_resume)
print(f"Candidate: {result.candidate_name}")
print(f"Score:     {result.overall_score}/10")
print(f"Tier:      {result.tier}")
print(f"Action:    {result.recommended_action}")
print(f"Matched:   {', '.join(result.skills_matched)}")
print(f"Missing:   {', '.join(result.skills_missing) or 'none'}")
print(f"Standout:  {result.standout}")
print(f"Concern:   {result.concern}")
print(f"\nPython type: {type(result)}")  # ResumeScore — not a dict or string

---

## Part 5 — Screen Multiple Resumes and Rank

---

The key power of a consistent schema: once every resume returns the same shape,
ranking is just a `sorted()` call. No parsing, no post-processing.

In [ ]:
RESUMES = [
    {
        "name": "Alex Kim",
        "text": """Alex Kim — Senior Software Engineer
7 years Python. Built REST APIs with FastAPI at two fintech startups.
Led a team of 3 engineers. PostgreSQL, Redis, AWS (ECS, RDS).
Deployed to production weekly using GitHub Actions + Docker.
Skills: Python, FastAPI, PostgreSQL, AWS, Docker, Pydantic, Git.""",
    },
    {
        "name": "Jordan Lee",
        "text": """Jordan Lee — Data Scientist
4 years total experience. Python, pandas, scikit-learn, Jupyter.
Built ML pipelines for e-commerce recommendation system.
Some Flask API work (internal tools only, never production).
No cloud deployment. Mostly data science and analytics.
Skills: Python, pandas, scikit-learn, SQL (basic), Flask.""",
    },
    {
        "name": "Sam Nguyen",
        "text": """Sam Nguyen — Backend Developer
2 years experience. Django REST Framework, PostgreSQL.
Junior role at a startup. Worked on user auth and CRUD endpoints.
Some GCP experience (App Engine). No team leadership.
Still learning Docker and CI/CD.
Skills: Python, Django, PostgreSQL, GCP (basic), Git.""",
    },
]

# Screen all resumes
results = []
for r in RESUMES:
    score = screener.invoke(r["text"])
    results.append(score)
    print(f"{score.candidate_name:15} | {score.overall_score}/10 | {score.tier}")

print("\nScreening complete.")

In [ ]:
# Rank by score — pure Python, no LLM involved
ranked = sorted(results, key=lambda x: x.overall_score, reverse=True)

print("RANKING")
print("-" * 50)
for i, r in enumerate(ranked, 1):
    action_icon = {"schedule_interview": "✓", "hold_for_review": "?", "pass": "✗"}.get(r.recommended_action, "?")
    print(f"{i}. {r.candidate_name:15} {r.overall_score}/10  {r.tier:12} {action_icon}")

print("\n--- Top candidate ---")
top = ranked[0]
print(f"Name:     {top.candidate_name}")
print(f"Standout: {top.standout}")
print(f"Missing:  {', '.join(top.skills_missing) or 'none'}")

---

## Exercise 1 — Add a `culture_fit_note` field

Add `culture_fit_note: str` to `ResumeScore` describing whether the candidate's
background fits an early-stage startup (fast-paced, wearing many hats).

Update the system prompt to ask the model to assess this.
Re-run on all three resumes.

**Why this matters:** Culture fit is often the hardest signal to extract from a resume.
Making it a required field forces the model to reason about it explicitly.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
from typing import List, Literal
from pydantic import BaseModel, Field

class ResumeScoreEx1(BaseModel):
    candidate_name: str
    overall_score: int = Field(ge=1, le=10)
    tier: Literal["strong_yes", "yes", "maybe", "no"]
    years_experience: int = Field(ge=0)
    skills_matched: List[str]
    skills_missing: List[str]
    standout: str
    concern: str
    recommended_action: Literal["schedule_interview", "hold_for_review", "pass"]
    # TODO: add culture_fit_note field

# TODO: update system prompt to assess startup culture fit
# TODO: re-run screener on all 3 resumes

In [ ]:
# ── Exercise 1 Answer Key ─────────────────────────────────────────────────────
from typing import List, Literal
from pydantic import BaseModel, Field

class ResumeScoreV2(BaseModel):
    candidate_name: str
    overall_score: int = Field(ge=1, le=10)
    tier: Literal["strong_yes", "yes", "maybe", "no"]
    years_experience: int = Field(ge=0)
    skills_matched: List[str]
    skills_missing: List[str]
    standout: str
    concern: str
    recommended_action: Literal["schedule_interview", "hold_for_review", "pass"]
    culture_fit_note: str = Field(
        description="One sentence on whether this person fits an early-stage startup environment"
    )

SYSTEM_V2 = SystemMessage(
    f"""{SYSTEM_PROMPT.content}

Also assess culture_fit_note: does this person fit an early-stage startup?
Look for signals like: startup experience, breadth of skills, adaptability, ownership mindset."""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
screener_v2 = SYSTEM_V2 | llm.with_structured_output(ResumeScoreV2)

for r in RESUMES:
    s = screener_v2.invoke(r["text"])
    print(f"{s.candidate_name:15} | {s.overall_score}/10 | {s.culture_fit_note}")

---

## What's Next?

You've built a rubric-driven screener that produces identical, sortable output for every input.

**The key pattern: schema as a comparison contract**
When you apply the same Pydantic schema to many inputs, the outputs are automatically comparable. This works for anything scored against a rubric: resumes, RFP responses, support tickets, product reviews.

**Next examples:**
- **7 — RFP Parser** — extract structured data from a long document (same schema idea, longer input)
- **8 — Multi-Agent Research** — one agent coordinates multiple specialist sub-agents

---
*Example 6 of 8 · agent-use-cases*